# 7. Large-Document RAG Evals & Experiments (Solution)

**Duration:** 75–100 minutes
**Core runtime:** offline, deterministic, CPU-only

One aggregate “RAG score” cannot tell you whether a PDF parser dropped a footnote, the
retriever missed one operand, the packer truncated an amendment, or the reader ignored
evidence it received. This module evaluates every boundary:

```text
parse → index → retrieve → rerank/expand → pack → answer → cite → abstain
```

## Learning objectives

1. Use immutable evidence/page labels rather than chunk IDs tied to one chunker.
2. Measure partial and strict all-evidence retrieval, rank quality, density, and budget.
3. Run a controlled factorial ablation and paired uncertainty estimate.
4. Slice by legal/financial failure mode and evidence position.
5. Test wrong-year/entity, amendments, OCR damage, missing evidence, and abstention.
6. Turn metrics into an explicit regression gate.


In [ ]:
from pathlib import Path
import json, math, random, re, statistics, time
from collections import Counter, defaultdict

EMBEDDED_BENCHMARK_JSON = r"""{"schema_version":"1.0.0","description":"Deterministic synthetic benchmark for evaluating retrieval, provenance, numerical reasoning, cross-reference resolution, and amendment handling over large financial and legal documents. Every company, person, agreement, amount, and event is fictional and is provided solely for testing.","blocks":[{"block_id":"FIN-001","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":1,"section_path":["Cover"],"content_type":"prose","text":"FICTIONAL TEST DOCUMENT. Northstar Systems Group, Inc. 2024 Annual Report. This document does not describe a real issuer and must not be used for investment decisions.","parent_section_id":null,"previous_block_id":null,"next_block_id":"FIN-002","references":[]},{"block_id":"FIN-002","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":2,"section_path":["Important Notices","Forward-Looking Statements"],"content_type":"prose","text":"Statements using expect, plan, may, or similar terms are forward-looking and subject to risks. The company undertakes no duty to update them except as required by fictional applicable law.","parent_section_id":"FIN-001","previous_block_id":"FIN-001","next_block_id":"FIN-003","references":["FIN-003"]},{"block_id":"FIN-003","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":4,"section_path":["Business","Company Overview"],"content_type":"prose","text":"Northstar provides cloud operations software and advisory services to mid-market enterprises. It reports Cloud Platform, Advisory Services, and Other as revenue categories.","parent_section_id":null,"previous_block_id":"FIN-002","next_block_id":"FIN-004","references":["FIN-008"]},{"block_id":"FIN-004","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":5,"section_path":["Business","Operating Model"],"content_type":"prose","text":"Cloud subscriptions are sold directly and through resellers. Advisory engagements include implementation and optimization work; Other primarily includes training and legacy maintenance.","parent_section_id":"FIN-003","previous_block_id":"FIN-003","next_block_id":"FIN-005","references":["FIN-008"]},{"block_id":"FIN-005","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":34,"section_path":["Management Discussion","Revenue Recognition Overview"],"content_type":"prose","text":"Subscription revenue is generally recognized ratably over the hosted-service term. Advisory revenue is recognized over time using labor hours as the measure of progress when enforceable payment rights exist.","parent_section_id":null,"previous_block_id":"FIN-004","next_block_id":"FIN-006","references":["FIN-042"]},{"block_id":"FIN-006","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":38,"section_path":["Financial Results","Consolidated Statements of Operations"],"content_type":"table","text":"USD millions | 2024 | 2023 | 2022\nRevenue | 842.6 | 771.4 | 699.1\nCost of revenue | 356.8 | 337.0 | 314.6\nGross profit | 485.8 | 434.4 | 384.5\nOperating income | 88.9 | 72.6 | 54.2\nNet income | 59.7 | 47.1 | 32.8","parent_section_id":null,"previous_block_id":"FIN-005","next_block_id":"FIN-007","references":["FIN-008","FIN-047"]},{"block_id":"FIN-007","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":39,"section_path":["Financial Results","Revenue Commentary"],"content_type":"prose","text":"Revenue increased by $71.2 million, or 9.2%, from 2023. Cloud Platform growth was partly offset by a modest decline in Advisory Services.","parent_section_id":"FIN-006","previous_block_id":"FIN-006","next_block_id":"FIN-008","references":["FIN-008"]},{"block_id":"FIN-008","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":40,"section_path":["Financial Results","Revenue by Category"],"content_type":"table","text":"USD millions | 2024 | 2023\nCloud Platform | 512.4 | 438.1\nAdvisory Services | 248.7 | 251.9\nOther | 81.5 | 81.4\nTotal revenue | 842.6 | 771.4","parent_section_id":"FIN-006","previous_block_id":"FIN-007","next_block_id":"FIN-009","references":["FIN-006","FIN-047"]},{"block_id":"FIN-009","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":41,"section_path":["Financial Results","Revenue by Geography"],"content_type":"table","text":"USD millions | 2024\nUnited States | 538.0\nCanada | 126.4\nUnited Kingdom | 94.2\nOther | 84.0\nTotal revenue | 842.6","parent_section_id":"FIN-006","previous_block_id":"FIN-008","next_block_id":"FIN-010","references":["FIN-006"]},{"block_id":"FIN-010","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":42,"section_path":["Financial Results","Contract Balances"],"content_type":"table","text":"USD millions | Dec. 31, 2024 | Dec. 31, 2023\nContract liabilities | 114.8 | 86.2\nUnbilled receivables | 31.6 | 27.9","parent_section_id":"FIN-006","previous_block_id":"FIN-009","next_block_id":"FIN-011","references":["FIN-011"]},{"block_id":"FIN-011","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":42,"section_path":["Financial Results","Remaining Performance Obligations"],"content_type":"prose","text":"Remaining performance obligations were $263.0 million at December 31, 2024. Northstar expects to recognize approximately 61% of that amount as revenue during the next 12 months and the remainder thereafter.","parent_section_id":"FIN-006","previous_block_id":"FIN-010","next_block_id":"FIN-012","references":["FIN-010"]},{"block_id":"FIN-012","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":43,"section_path":["Financial Results","Cost of Revenue"],"content_type":"prose","text":"Cost of revenue rose primarily because of hosting capacity and cloud-support payroll. Vendor credits reduced 2024 hosting expense by $2.1 million.","parent_section_id":"FIN-006","previous_block_id":"FIN-011","next_block_id":"FIN-013","references":[]},{"block_id":"FIN-013","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":44,"section_path":["Financial Results","Workforce"],"content_type":"prose","text":"Year-end headcount was 2,140, compared with 2,015 a year earlier. These figures include part-time employees but exclude independent contractors.","parent_section_id":"FIN-006","previous_block_id":"FIN-012","next_block_id":"FIN-014","references":[]},{"block_id":"FIN-014","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":45,"section_path":["Financial Results","Operating Expenses"],"content_type":"table","text":"USD millions | 2024 | 2023\nResearch and development | 112.2 | 101.4\nSales and marketing | 139.5 | 131.2\nGeneral and administrative | 78.7 | 76.5\nRestructuring | 6.4 | 4.1","parent_section_id":"FIN-006","previous_block_id":"FIN-013","next_block_id":"FIN-015","references":["FIN-015"]},{"block_id":"FIN-015","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":46,"section_path":["Financial Results","Restructuring"],"content_type":"footnote","text":"The $6.4 million 2024 restructuring charge consisted of $5.1 million of severance and $1.3 million of office-exit costs. No material cash charges are expected after 2025.","parent_section_id":"FIN-014","previous_block_id":"FIN-014","next_block_id":"FIN-016","references":["FIN-014"]},{"block_id":"FIN-016","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":47,"section_path":["Financial Results","Foreign Currency"],"content_type":"prose","text":"Currency movements reduced reported 2024 revenue growth by approximately 0.6 percentage points. This estimate is management's constant-currency analysis and is not a GAAP measure.","parent_section_id":"FIN-006","previous_block_id":"FIN-015","next_block_id":"FIN-017","references":[]},{"block_id":"FIN-017","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":48,"section_path":["Financial Results","Non-GAAP Reconciliation"],"content_type":"table","text":"USD millions | 2024\nNet income | 59.7\nIncome tax expense | 18.7\nInterest expense, net | 8.4\nDepreciation and amortization | 21.8\nAdjusted EBITDA | 108.6","parent_section_id":"FIN-006","previous_block_id":"FIN-016","next_block_id":"FIN-018","references":["FIN-033"]},{"block_id":"FIN-018","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":50,"section_path":["Liquidity","Cash Flow Summary"],"content_type":"table","text":"USD millions | 2024 | 2023\nNet cash provided by operating activities | 132.1 | 110.8\nCapital expenditures | (34.7) | (29.2)\nAcquisition payments, net of cash acquired | (12.0) | (4.5)\nFree cash flow | 97.4 | 81.6","parent_section_id":null,"previous_block_id":"FIN-017","next_block_id":"FIN-019","references":["FIN-019","FIN-047"]},{"block_id":"FIN-019","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":51,"section_path":["Liquidity","Cash Flow Commentary"],"content_type":"prose","text":"Free cash flow is defined as operating cash flow less capital expenditures. The 2024 increase mainly reflected higher collections, partially offset by payroll timing.","parent_section_id":"FIN-018","previous_block_id":"FIN-018","next_block_id":"FIN-020","references":["FIN-018"]},{"block_id":"FIN-020","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":53,"section_path":["Liquidity","Cash and Borrowings"],"content_type":"table","text":"USD millions | Dec. 31, 2024\nCash and cash equivalents | 96.3\nRevolving credit facility drawn | 45.0\nTerm loan principal | 120.0\nTotal debt principal | 165.0","parent_section_id":"FIN-018","previous_block_id":"FIN-019","next_block_id":"FIN-021","references":["FIN-024"]},{"block_id":"FIN-021","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":54,"section_path":["Liquidity","Liquidity Outlook"],"content_type":"prose","text":"Management believes existing cash, operating cash flows, and $155.0 million of undrawn revolver capacity will fund operations and contractual obligations for at least twelve months.","parent_section_id":"FIN-018","previous_block_id":"FIN-020","next_block_id":"FIN-022","references":["FIN-020"]},{"block_id":"FIN-022","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":56,"section_path":["Liquidity","Debt Covenants"],"content_type":"clause","text":"The credit agreement requires a consolidated net leverage ratio no greater than 3.50 to 1.00 and a consolidated interest coverage ratio no less than 3.00 to 1.00, each tested quarterly.","parent_section_id":"FIN-018","previous_block_id":"FIN-021","next_block_id":"FIN-023","references":["FIN-023"]},{"block_id":"FIN-023","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":56,"section_path":["Liquidity","Debt Covenants","Compliance"],"content_type":"prose","text":"At December 31, 2024, Northstar's net leverage ratio was 1.42 to 1.00 and its interest coverage ratio was 8.10 to 1.00. Northstar was in compliance with all financial covenants.","parent_section_id":"FIN-022","previous_block_id":"FIN-022","next_block_id":"FIN-024","references":["FIN-022"]},{"block_id":"FIN-024","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":58,"section_path":["Liquidity","Debt Maturities"],"content_type":"table","text":"USD millions | Principal due\n2025 | 10.0\n2026 | 15.0\n2027 | 60.0\n2028 | 80.0\nThereafter | 0.0\nTotal | 165.0","parent_section_id":"FIN-018","previous_block_id":"FIN-023","next_block_id":"FIN-025","references":["FIN-020"]},{"block_id":"FIN-025","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":59,"section_path":["Liquidity","Interest Rates"],"content_type":"footnote","text":"The term loan bears interest at synthetic SOFR plus 2.00%. The revolver bears interest at synthetic SOFR plus a margin ranging from 1.50% to 2.25% based on leverage.","parent_section_id":"FIN-018","previous_block_id":"FIN-024","next_block_id":"FIN-026","references":["FIN-026"]},{"block_id":"FIN-026","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":60,"section_path":["Market Risk","Interest Rate Sensitivity"],"content_type":"prose","text":"A hypothetical 100-basis-point increase in variable rates would increase annual pre-tax interest expense by approximately $1.7 million, assuming year-end borrowings remained outstanding.","parent_section_id":null,"previous_block_id":"FIN-025","next_block_id":"FIN-027","references":["FIN-020","FIN-025"]},{"block_id":"FIN-027","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":65,"section_path":["Financial Statements","Consolidated Balance Sheets"],"content_type":"table","text":"USD millions | 2024 | 2023\nCash and cash equivalents | 96.3 | 82.7\nAccounts receivable, net | 142.5 | 129.6\nTotal assets | 781.4 | 716.2\nTotal debt, net | 161.8 | 174.9\nTotal stockholders' equity | 338.7 | 296.5","parent_section_id":null,"previous_block_id":"FIN-026","next_block_id":"FIN-028","references":["FIN-020"]},{"block_id":"FIN-028","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":78,"section_path":["Notes","Accounts Receivable"],"content_type":"table","text":"USD millions | 2024 | 2023\nGross accounts receivable | 148.2 | 134.3\nAllowance for credit losses | (5.7) | (4.7)\nAccounts receivable, net | 142.5 | 129.6","parent_section_id":null,"previous_block_id":"FIN-027","next_block_id":"FIN-029","references":["FIN-029"]},{"block_id":"FIN-029","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":79,"section_path":["Notes","Accounts Receivable","Credit Losses"],"content_type":"prose","text":"The allowance increased because of aging in two advisory accounts. Write-offs were $1.1 million in 2024; recoveries were immaterial.","parent_section_id":"FIN-028","previous_block_id":"FIN-028","next_block_id":"FIN-030","references":["FIN-028"]},{"block_id":"FIN-030","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":82,"section_path":["Notes","Goodwill"],"content_type":"table","text":"USD millions | Cloud Platform | Advisory Services | Total\nGoodwill at Dec. 31, 2024 | 132.0 | 38.5 | 170.5","parent_section_id":null,"previous_block_id":"FIN-029","next_block_id":"FIN-031","references":["FIN-031"]},{"block_id":"FIN-031","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":83,"section_path":["Notes","Business Combinations"],"content_type":"prose","text":"Northstar acquired fictional telemetry vendor Quill Harbor Labs on July 8, 2024 for $24.0 million, including $12.0 million paid in cash at closing and contingent consideration.","parent_section_id":null,"previous_block_id":"FIN-030","next_block_id":"FIN-032","references":["FIN-030"]},{"block_id":"FIN-032","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":85,"section_path":["Notes","Intangible Assets"],"content_type":"table","text":"USD millions | Gross carrying amount | Accumulated amortization | Net\nDeveloped technology | 54.0 | (18.6) | 35.4\nCustomer relationships | 31.5 | (9.9) | 21.6\nTrade names | 6.0 | (2.0) | 4.0","parent_section_id":null,"previous_block_id":"FIN-031","next_block_id":"FIN-033","references":[]},{"block_id":"FIN-033","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":91,"section_path":["Notes","Income Taxes"],"content_type":"table","text":"USD millions except rates | 2024 | 2023\nIncome before income taxes | 78.4 | 62.2\nIncome tax expense | 18.7 | 15.1\nEffective tax rate | 23.8% | 24.3%","parent_section_id":null,"previous_block_id":"FIN-032","next_block_id":"FIN-034","references":["FIN-017"]},{"block_id":"FIN-034","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":92,"section_path":["Notes","Income Taxes","Rate Reconciliation"],"content_type":"footnote","text":"The 2024 effective rate differed from the fictional federal statutory rate primarily because of provincial taxes, research credits, and nondeductible executive compensation.","parent_section_id":"FIN-033","previous_block_id":"FIN-033","next_block_id":"FIN-035","references":["FIN-033"]},{"block_id":"FIN-035","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":96,"section_path":["Notes","Leases"],"content_type":"table","text":"USD millions | 2025 | 2026 | 2027 | 2028 | Thereafter\nOperating lease payments | 14.2 | 12.8 | 10.4 | 8.1 | 16.5","parent_section_id":null,"previous_block_id":"FIN-034","next_block_id":"FIN-036","references":[]},{"block_id":"FIN-036","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":98,"section_path":["Notes","Commitments"],"content_type":"prose","text":"Non-cancellable hosting commitments total $72.0 million through 2028. Purchase orders cancellable without significant penalty are excluded.","parent_section_id":null,"previous_block_id":"FIN-035","next_block_id":"FIN-037","references":[]},{"block_id":"FIN-037","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":101,"section_path":["Notes","Contingencies","Orion Claim"],"content_type":"footnote","text":"A fictional former reseller, Orion Vale LLC, alleges breach of contract. Management estimates a reasonably possible loss range of $2.5 million to $4.0 million and has accrued $2.5 million. No amount within the range above the accrual is considered a better estimate.","parent_section_id":null,"previous_block_id":"FIN-036","next_block_id":"FIN-038","references":[]},{"block_id":"FIN-038","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":104,"section_path":["Notes","Subsequent Events"],"content_type":"prose","text":"On January 20, 2025, Northstar signed a non-binding letter of intent regarding a small analytics business. No acquisition had closed when this synthetic report was issued.","parent_section_id":null,"previous_block_id":"FIN-037","next_block_id":"FIN-039","references":[]},{"block_id":"FIN-039","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":106,"section_path":["Risk Management","Cybersecurity"],"content_type":"prose","text":"The board receives quarterly cyber-risk reporting. Northstar experienced routine unsuccessful attempts but identified no incident during 2024 that materially affected operations.","parent_section_id":null,"previous_block_id":"FIN-038","next_block_id":"FIN-040","references":[]},{"block_id":"FIN-040","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":108,"section_path":["Controls","Management Assessment"],"content_type":"prose","text":"Management concluded that disclosure controls and internal control over financial reporting were effective as of December 31, 2024. No material weakness was identified.","parent_section_id":null,"previous_block_id":"FIN-039","next_block_id":"FIN-041","references":["FIN-041"]},{"block_id":"FIN-041","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":110,"section_path":["Independent Auditor Report","Opinion"],"content_type":"prose","text":"Fictional audit firm Lantern & Moss LLP states that the synthetic financial statements present fairly, in all material respects, the fictional company's financial position and results under the stated test framework.","parent_section_id":null,"previous_block_id":"FIN-040","next_block_id":"FIN-042","references":["FIN-040"]},{"block_id":"FIN-042","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":112,"section_path":["Independent Auditor Report","Critical Audit Matter"],"content_type":"prose","text":"The sole critical audit matter was evaluating revenue recognition for advisory arrangements with variable consideration, including testing contract terms and management's estimates of progress.","parent_section_id":"FIN-041","previous_block_id":"FIN-041","next_block_id":"FIN-043","references":["FIN-005"]},{"block_id":"FIN-043","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":115,"section_path":["Supplemental Information","Environmental Metrics"],"content_type":"table","text":"Operational metric | 2024 | 2023\nScope 2 market-based emissions, metric tons CO2e | 7,800 | 8,250\nRenewable electricity coverage | 68% | 61%","parent_section_id":null,"previous_block_id":"FIN-042","next_block_id":"FIN-044","references":[]},{"block_id":"FIN-044","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":117,"section_path":["Governance","Board"],"content_type":"prose","text":"The fictional board has eight directors, seven of whom management classifies as independent. The audit committee met six times during 2024.","parent_section_id":null,"previous_block_id":"FIN-043","next_block_id":"FIN-045","references":[]},{"block_id":"FIN-045","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":119,"section_path":["Shareholder Information"],"content_type":"prose","text":"The company had approximately 420 holders of record at February 14, 2025. This count excludes beneficial owners holding through brokers.","parent_section_id":null,"previous_block_id":"FIN-044","next_block_id":"FIN-046","references":[]},{"block_id":"FIN-046","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":121,"section_path":["Appendix","Definitions"],"content_type":"definition","text":"For this synthetic report, ARR means annualized recurring subscription revenue at period end; it is an operating metric and is not total revenue under the test accounting framework.","parent_section_id":null,"previous_block_id":"FIN-045","next_block_id":"FIN-047","references":["FIN-006"]},{"block_id":"FIN-047","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":122,"section_path":["Appendix","Rounding Convention"],"content_type":"footnote","text":"Amounts are in U.S. dollars and in millions unless stated otherwise. Columns may not sum because of rounding; percentages are calculated from unrounded amounts.","parent_section_id":"FIN-046","previous_block_id":"FIN-046","next_block_id":"FIN-048","references":["FIN-006","FIN-008","FIN-018"]},{"block_id":"FIN-048","doc_id":"northstar_2024_annual_report","document_title":"Northstar Systems Group 2024 Annual Report (Fictional)","document_version":"Final synthetic filing","as_of_date":"2024-12-31","page":124,"section_path":["Back Cover"],"content_type":"prose","text":"End of fictional Northstar Systems Group 2024 Annual Report. Corporate address and investor contact details are intentionally omitted from this benchmark.","parent_section_id":null,"previous_block_id":"FIN-047","next_block_id":null,"references":["FIN-001"]},{"block_id":"MSA-001","doc_id":"northstar_redwood_msa","document_title":"Master Services Agreement between Northstar Systems Group and Redwood Harbor Bank (Fictional)","document_version":"Original executed version","as_of_date":"2023-04-01","page":1,"section_path":["Title"],"content_type":"clause","text":"FICTIONAL TEST AGREEMENT. Master Services Agreement dated April 1, 2023. It creates no rights or obligations for any real person or entity.","parent_section_id":null,"previous_block_id":null,"next_block_id":"MSA-002","references":[]},{"block_id":"MSA-002","doc_id":"northstar_redwood_msa","document_title":"Master Services Agreement between Northstar Systems Group and Redwood Harbor Bank (Fictional)","document_version":"Original executed version","as_of_date":"2023-04-01","page":1,"section_path":["Preamble","Parties and Effective Date"],"content_type":"clause","text":"This Agreement is entered into by Northstar Systems Group, Inc. as Provider and Redwood Harbor Bank, N.A. as Customer, effective April 1, 2023.","parent_section_id":"MSA-001","previous_block_id":"MSA-001","next_block_id":"MSA-003","references":["AMD-002"]},{"block_id":"MSA-003","doc_id":"northstar_redwood_msa","document_title":"Master Services Agreement between Northstar Systems Group and Redwood Harbor Bank (Fictional)","document_version":"Original executed version","as_of_date":"2023-04-01","page":2,"section_path":["Recitals"],"content_type":"prose","text":"Customer wishes to obtain hosted operations software and related professional services, and Provider wishes to supply those services under statements of work.","parent_section_id":"MSA-001","previous_block_id":"MSA-002","next_block_id":"MSA-004","references":["MSA-007"]},{"block_id":"MSA-004","doc_id":"northstar_redwood_msa","document_title":"Master Services Agreement between Northstar Systems Group and Redwood Harbor Bank (Fictional)","document_version":"Original executed version","as_of_date":"2023-04-01","page":3,"section_path":["1. Definitions","1.1 Affiliate"],"content_type":"definition","text":"Affiliate means an entity that directly or indirectly controls, is controlled by, or is under common control with a party, where control means ownership of more than fifty percent of voting interests.","parent_section_id":null,"previous_block_id":"MSA-003","next_block_id":"MSA-005","references":[]},{"block_id":"MSA-005","doc_id":"northstar_redwood_msa","document_title":"Master Services Agreement between Northstar Systems Group and Redwood Harbor Bank (Fictional)","document_version":"Original executed version","as_of_date":"2023-04-01","page":3,"section_path":["1. Definitions","1.2 Confidential Information"],"content_type":"definition","text":"Confidential Information means non-public information disclosed by or for a party that is marked confidential or reasonably should be understood as confidential given its nature and the circumstances.","parent_section_id":"MSA-004","previous_block_id":"MSA-004","next_block_id":"MSA-006","references":["MSA-019","MSA-020"]},{"block_id":"MSA-006","doc_id":"northstar_redwood_msa","document_title":"Master Services Agreement between Northstar Systems Group and Redwood Harbor Bank (Fictional)","document_version":"Original executed version","as_of_date":"2023-04-01","page":4,"section_path":["1. Definitions","1.3 Fees"],"content_type":"definition","text":"Fees means the charges expressly stated in an executed statement of work, excluding taxes and approved reimbursable expenses.","parent_section_id":"MSA-004","previous_block_id":"MSA-005","next_block_id":"MSA-007","references":["MSA-011","MSA-029"]},{"block_id":"MSA-007","doc_id":"northstar_redwood_msa","document_title":"Master Services Agreement between Northstar Systems Group and Redwood Harbor Bank (Fictional)","document_version":"Original executed version","as_of_date":"2023-04-01","page":5,"section_path":["2. Services","2.1 Statements of Work"],"content_type":"clause","text":"Provider will perform only services described in a statement of work signed by authorized representatives of both parties. A purchase order does not modify this Agreement.","parent_section_id":null,"previous_block_id":"MSA-006","next_block_id":"MSA-008","references":["MSA-008"]},{"block_id":"MSA-008","doc_id":"northstar_redwood_msa","document_title":"Master Services Agreement between Northstar Systems Group and Redwood Harbor Bank (Fictional)","document_version":"Original executed version","as_of_date":"2023-04-01","page":5,"section_path":["2. Services","2.2 Order of Precedence"],"content_type":"clause","text":"If documents conflict, an executed statement of work controls only for its specific business terms, and this Agreement controls for legal terms unless the statement expressly identifies the section being overridden. A later signed amendment controls over both.","parent_section_id":"MSA-007","previous_block_id":"MSA-007","next_block_id":"MSA-009","references":["AMD-003"]},{"block_id":"MSA-009","doc_id":"northstar_redwood_msa","document_title":"Master Services Agreement between Northstar Systems Group and Redwood Harbor Bank (Fictional)","document_version":"Original executed version","as_of_date":"2023-04-01","page":6,"section_path":["2. Services","2.3 Performance Standard"],"content_type":"clause","text":"Provider will perform professional services in a professional and workmanlike manner using personnel with appropriate skill and experience.","parent_section_id":"MSA-007","previous_block_id":"MSA-008","next_block_id":"MSA-010","references":[]},{"block_id":"MSA-010","doc_id":"northstar_redwood_msa","document_title":"Master Services Agreement between Northstar Systems Group and Redwood Harbor Bank (Fictional)","document_version":"Original executed version","as_of_date":"2023-04-01","page":7,"section_path":["3. Acceptance","3.1 Review"],"content_type":"clause","text":"Customer must test each stated deliverable and give written notice describing material nonconformities within ten business days after delivery. A deliverable is deemed accepted if Customer gives no timely notice.","parent_section_id":null,"previous_block_id":"MSA-009","next_block_id":"MSA-011","references":[]},{"block_id":"MSA-011","doc_id":"northstar_redwood_msa","document_title":"Master Services Agreement between Northstar Systems Group and Redwood Harbor Bank (Fictional)","document_version":"Original executed version","as_of_date":"2023-04-01","page":8,"section_path":["4. Fees","4.1 Invoices"],"content_type":"clause","text":"Undisputed invoice amounts are due thirty calendar days from Customer's receipt of a correct invoice. Customer must notify Provider of a good-faith dispute before the due date.","parent_section_id":null,"previous_block_id":"MSA-010","next_block_id":"MSA-012","references":["MSA-006"]},{"block_id":"MSA-012","doc_id":"northstar_redwood_msa","document_title":"Master Services Agreement between Northstar Systems Group and Redwood Harbor Bank (Fictional)","document_version":"Original executed version","as_of_date":"2023-04-01","page":8,"section_path":["4. Fees","4.2 Late Charges"],"content_type":"clause","text":"Overdue undisputed amounts accrue a late charge of 1.0% per month or the maximum lawful rate, whichever is lower, from the due date until paid.","parent_section_id":"MSA-011","previous_block_id":"MSA-011","next_block_id":"MSA-013","references":[]},{"block_id":"MSA-013","doc_id":"northstar_redwood_msa","document_title":"Master Services Agreement between Northstar Systems Group and Redwood Harbor Bank (Fictional)","document_version":"Original executed version","as_of_date":"2023-04-01","page":9,"section_path":["4. Fees","4.3 Taxes"],"content_type":"clause","text":"Customer is responsible for transaction taxes on the services, except taxes based on Provider's net income, property, or personnel.","parent_section_id":"MSA-011","previous_block_id":"MSA-012","next_block_id":"MSA-014","references":[]},{"block_id":"MSA-014","doc_id":"northstar_redwood_msa","document_title":"Master Services Agreement between Northstar Systems Group and Redwood Harbor Bank (Fictional)","document_version":"Original executed version","as_of_date":"2023-04-01","page":9,"section_path":["4. Fees","4.4 Expenses"],"content_type":"clause","text":"Customer will reimburse reasonable travel expenses only if the applicable statement of work permits them and Customer approved them in writing before they were incurred.","parent_section_id":"MSA-011","previous_block_id":"MSA-013","next_block_id":"MSA-015","references":[]},{"block_id":"MSA-015","doc_id":"northstar_redwood_msa","document_title":"Master Services Agreement between Northstar Systems Group and Redwood Harbor Bank (Fictional)","document_version":"Original executed version","as_of_date":"2023-04-01","page":10,"section_path":["5. Term and Termination","5.1 Term"],"content_type":"clause","text":"The initial term begins on the Effective Date and continues for three years. It then renews for successive one-year periods unless either party gives notice of non-renewal at least sixty days before renewal.","parent_section_id":null,"previous_block_id":"MSA-014","next_block_id":"MSA-016","references":["MSA-002"]},{"block_id":"MSA-016","doc_id":"northstar_redwood_msa","document_title":"Master Services Agreement between Northstar Systems Group and Redwood Harbor Bank (Fictional)","document_version":"Original executed version","as_of_date":"2023-04-01","page":10,"section_path":["5. Term and Termination","5.2 Convenience"],"content_type":"clause","text":"Customer may terminate this Agreement or a statement of work for convenience on ninety days' prior written notice and must pay Fees for services performed through the termination date.","parent_section_id":"MSA-015","previous_block_id":"MSA-015","next_block_id":"MSA-017","references":["AMD-007"]},{"block_id":"MSA-017","doc_id":"northstar_redwood_msa","document_title":"Master Services Agreement between Northstar Systems Group and Redwood Harbor Bank (Fictional)","document_version":"Original executed version","as_of_date":"2023-04-01","page":11,"section_path":["5. Term and Termination","5.3 Cause"],"content_type":"clause","text":"Either party may terminate for material breach not cured within thirty days after written notice, except that the cure period for failure to pay an undisputed amount is ten days.","parent_section_id":"MSA-015","previous_block_id":"MSA-016","next_block_id":"MSA-018","references":[]},{"block_id":"MSA-018","doc_id":"northstar_redwood_msa","document_title":"Master Services Agreement between Northstar Systems Group and Redwood Harbor Bank (Fictional)","document_version":"Original executed version","as_of_date":"2023-04-01","page":11,"section_path":["5. Term and Termination","5.4 Effect"],"content_type":"clause","text":"Upon termination, Customer will cease using the services and pay accrued undisputed Fees. Provisions that by nature should survive, including confidentiality, intellectual property, liability, and dispute terms, will survive.","parent_section_id":"MSA-015","previous_block_id":"MSA-017","next_block_id":"MSA-019","references":["MSA-019","MSA-025","MSA-029"]},{"block_id":"MSA-019","doc_id":"northstar_redwood_msa","document_title":"Master Services Agreement between Northstar Systems Group and Redwood Harbor Bank (Fictional)","document_version":"Original executed version","as_of_date":"2023-04-01","page":12,"section_path":["6. Confidentiality","6.1 Protection and Duration"],"content_type":"clause","text":"A receiving party will protect Confidential Information using at least reasonable care and use it only to perform this Agreement. These duties continue for five years after disclosure; trade secrets remain protected for so long as they qualify as trade secrets under applicable law.","parent_section_id":null,"previous_block_id":"MSA-018","next_block_id":"MSA-020","references":["MSA-005"]},{"block_id":"MSA-020","doc_id":"northstar_redwood_msa","document_title":"Master Services Agreement between Northstar Systems Group and Redwood Harbor Bank (Fictional)","document_version":"Original executed version","as_of_date":"2023-04-01","page":12,"section_path":["6. Confidentiality","6.2 Exclusions"],"content_type":"clause","text":"Confidential Information excludes information the recipient can document was lawfully known without restriction, independently developed, rightfully received from another source, or publicly available without breach.","parent_section_id":"MSA-019","previous_block_id":"MSA-019","next_block_id":"MSA-021","references":["MSA-005"]},{"block_id":"MSA-021","doc_id":"northstar_redwood_msa","document_title":"Master Services Agreement between Northstar Systems Group and Redwood Harbor Bank (Fictional)","document_version":"Original executed version","as_of_date":"2023-04-01","page":13,"section_path":["6. Confidentiality","6.3 Required Disclosure"],"content_type":"clause","text":"If legally compelled to disclose, the recipient will, where lawful, give prompt notice and reasonable assistance so the discloser may seek protection. Only the legally required portion may be disclosed.","parent_section_id":"MSA-019","previous_block_id":"MSA-020","next_block_id":"MSA-022","references":[]},{"block_id":"MSA-022","doc_id":"northstar_redwood_msa","document_title":"Master Services Agreement between Northstar Systems Group and Redwood Harbor Bank (Fictional)","document_version":"Original executed version","as_of_date":"2023-04-01","page":14,"section_path":["7. Security","7.1 Safeguards"],"content_type":"clause","text":"Provider will maintain a written security program with administrative, technical, and physical safeguards appropriate to the sensitivity of Customer Data, including access control, encryption in transit, and security training.","parent_section_id":null,"previous_block_id":"MSA-021","next_block_id":"MSA-023","references":["MSA-023"]},{"block_id":"MSA-023","doc_id":"northstar_redwood_msa","document_title":"Master Services Agreement between Northstar Systems Group and Redwood Harbor Bank (Fictional)","document_version":"Original executed version","as_of_date":"2023-04-01","page":15,"section_path":["7. Security","7.2 Security Incident"],"content_type":"clause","text":"Provider will notify Customer without undue delay and in no event later than forty-eight hours after confirming unauthorized access to Customer Data, and will provide material updates and reasonable remediation cooperation.","parent_section_id":"MSA-022","previous_block_id":"MSA-022","next_block_id":"MSA-024","references":["AMD-004"]},{"block_id":"MSA-024","doc_id":"northstar_redwood_msa","document_title":"Master Services Agreement between Northstar Systems Group and Redwood Harbor Bank (Fictional)","document_version":"Original executed version","as_of_date":"2023-04-01","page":16,"section_path":["7. Security","7.3 Data Use"],"content_type":"clause","text":"Provider may process Customer Data only to provide, secure, support, and improve the contracted services. Aggregated data may be used only if it cannot reasonably identify Customer or an individual.","parent_section_id":"MSA-022","previous_block_id":"MSA-023","next_block_id":"MSA-025","references":["AMD-008"]},{"block_id":"MSA-025","doc_id":"northstar_redwood_msa","document_title":"Master Services Agreement between Northstar Systems Group and Redwood Harbor Bank (Fictional)","document_version":"Original executed version","as_of_date":"2023-04-01","page":18,"section_path":["8. Intellectual Property","8.1 Ownership"],"content_type":"clause","text":"Provider retains its pre-existing technology and general know-how. Customer retains Customer Data. Bespoke deliverables identified as work made for hire in a statement of work belong to Customer upon full payment.","parent_section_id":null,"previous_block_id":"MSA-024","next_block_id":"MSA-026","references":[]},{"block_id":"MSA-026","doc_id":"northstar_redwood_msa","document_title":"Master Services Agreement between Northstar Systems Group and Redwood Harbor Bank (Fictional)","document_version":"Original executed version","as_of_date":"2023-04-01","page":18,"section_path":["8. Intellectual Property","8.2 License"],"content_type":"clause","text":"During an applicable subscription term, Provider grants Customer a non-exclusive, non-transferable right for authorized users to access and use the hosted service for internal business purposes.","parent_section_id":"MSA-025","previous_block_id":"MSA-025","next_block_id":"MSA-027","references":[]},{"block_id":"MSA-027","doc_id":"northstar_redwood_msa","document_title":"Master Services Agreement between Northstar Systems Group and Redwood Harbor Bank (Fictional)","document_version":"Original executed version","as_of_date":"2023-04-01","page":19,"section_path":["8. Intellectual Property","8.3 Feedback"],"content_type":"clause","text":"Customer may provide feedback voluntarily. Provider may use feedback without restriction, provided it does not disclose Customer Confidential Information.","parent_section_id":"MSA-025","previous_block_id":"MSA-026","next_block_id":"MSA-028","references":["MSA-019"]},{"block_id":"MSA-028","doc_id":"northstar_redwood_msa","document_title":"Master Services Agreement between Northstar Systems Group and Redwood Harbor Bank (Fictional)","document_version":"Original executed version","as_of_date":"2023-04-01","page":21,"section_path":["9. Indemnification"],"content_type":"clause","text":"Provider will defend Customer against a third-party claim that the hosted service infringes a patent, copyright, or trademark, and pay finally awarded damages, subject to prompt notice, control of defense, and cooperation.","parent_section_id":null,"previous_block_id":"MSA-027","next_block_id":"MSA-029","references":["MSA-029","AMD-005"]},{"block_id":"MSA-029","doc_id":"northstar_redwood_msa","document_title":"Master Services Agreement between Northstar Systems Group and Redwood Harbor Bank (Fictional)","document_version":"Original executed version","as_of_date":"2023-04-01","page":23,"section_path":["10. Limitation of Liability"],"content_type":"clause","text":"Except for excluded claims, each party's aggregate liability is capped at Fees paid or payable under the affected statements of work during the twelve months before the event. Liability for confidentiality breach or violation of security obligations is capped at two times that amount. Fraud, willful misconduct, and Provider's intellectual-property indemnity are excluded from all caps.","parent_section_id":null,"previous_block_id":"MSA-028","next_block_id":"MSA-030","references":["MSA-006","MSA-028","AMD-005"]},{"block_id":"MSA-030","doc_id":"northstar_redwood_msa","document_title":"Master Services Agreement between Northstar Systems Group and Redwood Harbor Bank (Fictional)","document_version":"Original executed version","as_of_date":"2023-04-01","page":25,"section_path":["11. General","11.8 Governing Law"],"content_type":"clause","text":"The laws of Ontario and the federal laws of Canada applicable there govern, without conflict-of-law rules. Courts located in Toronto, Ontario have exclusive jurisdiction. The parties exclude the U.N. Convention on Contracts for the International Sale of Goods.","parent_section_id":null,"previous_block_id":"MSA-029","next_block_id":null,"references":[]},{"block_id":"AMD-001","doc_id":"northstar_redwood_msa_amendment_1","document_title":"Amendment No. 1 to Northstar-Redwood Master Services Agreement (Fictional)","document_version":"Executed Amendment No. 1","as_of_date":"2024-09-15","page":1,"section_path":["Title"],"content_type":"amendment","text":"FICTIONAL TEST AMENDMENT. Amendment No. 1 to the April 1, 2023 Master Services Agreement. It has no legal effect outside this benchmark.","parent_section_id":null,"previous_block_id":null,"next_block_id":"AMD-002","references":["MSA-001"]},{"block_id":"AMD-002","doc_id":"northstar_redwood_msa_amendment_1","document_title":"Amendment No. 1 to Northstar-Redwood Master Services Agreement (Fictional)","document_version":"Executed Amendment No. 1","as_of_date":"2024-09-15","page":1,"section_path":["Preamble"],"content_type":"amendment","text":"Northstar Systems Group, Inc. and Redwood Harbor Bank, N.A. enter this Amendment effective September 15, 2024. Capitalized terms not defined here have the meanings in the Agreement.","parent_section_id":"AMD-001","previous_block_id":"AMD-001","next_block_id":"AMD-003","references":["MSA-002"]},{"block_id":"AMD-003","doc_id":"northstar_redwood_msa_amendment_1","document_title":"Amendment No. 1 to Northstar-Redwood Master Services Agreement (Fictional)","document_version":"Executed Amendment No. 1","as_of_date":"2024-09-15","page":2,"section_path":["1. Construction","1.1 Priority"],"content_type":"amendment","text":"If this Amendment conflicts with the Agreement or any statement of work, this Amendment controls. Except as expressly modified, the Agreement remains unchanged.","parent_section_id":null,"previous_block_id":"AMD-002","next_block_id":"AMD-004","references":["MSA-008"]},{"block_id":"AMD-004","doc_id":"northstar_redwood_msa_amendment_1","document_title":"Amendment No. 1 to Northstar-Redwood Master Services Agreement (Fictional)","document_version":"Executed Amendment No. 1","as_of_date":"2024-09-15","page":2,"section_path":["2. Security Incident Notice"],"content_type":"amendment","text":"Section 7.2 is amended by replacing 'forty-eight hours' with 'twenty-four hours.' Provider must therefore notify Customer no later than twenty-four hours after confirming unauthorized access to Customer Data.","parent_section_id":null,"previous_block_id":"AMD-003","next_block_id":"AMD-005","references":["MSA-023"]},{"block_id":"AMD-005","doc_id":"northstar_redwood_msa_amendment_1","document_title":"Amendment No. 1 to Northstar-Redwood Master Services Agreement (Fictional)","document_version":"Executed Amendment No. 1","as_of_date":"2024-09-15","page":3,"section_path":["3. Limitation of Liability"],"content_type":"amendment","text":"Section 10 is replaced. The general aggregate cap is one and one-half times Fees paid or payable under affected statements of work during the twelve months before the event. The cap for confidentiality breach or violation of security obligations is three times that amount. Fraud, willful misconduct, and Provider's intellectual-property indemnity remain uncapped.","parent_section_id":null,"previous_block_id":"AMD-004","next_block_id":"AMD-006","references":["MSA-028","MSA-029"]},{"block_id":"AMD-006","doc_id":"northstar_redwood_msa_amendment_1","document_title":"Amendment No. 1 to Northstar-Redwood Master Services Agreement (Fictional)","document_version":"Executed Amendment No. 1","as_of_date":"2024-09-15","page":3,"section_path":["4. Fees","4.1 Platform Fee"],"content_type":"amendment","text":"Beginning October 1, 2024, the annual platform Fee under Statement of Work 1 is $480,000, invoiced quarterly in equal installments. Usage charges, taxes, and expenses are separate.","parent_section_id":null,"previous_block_id":"AMD-005","next_block_id":"AMD-007","references":["MSA-006","MSA-011"]},{"block_id":"AMD-007","doc_id":"northstar_redwood_msa_amendment_1","document_title":"Amendment No. 1 to Northstar-Redwood Master Services Agreement (Fictional)","document_version":"Executed Amendment No. 1","as_of_date":"2024-09-15","page":4,"section_path":["5. Termination for Convenience"],"content_type":"amendment","text":"For any renewal period beginning after this Amendment's effective date, Section 5.2 is replaced so Customer may terminate for convenience on sixty days' prior written notice. The original ninety-day period continues during the initial term.","parent_section_id":null,"previous_block_id":"AMD-006","next_block_id":"AMD-008","references":["MSA-015","MSA-016"]},{"block_id":"AMD-008","doc_id":"northstar_redwood_msa_amendment_1","document_title":"Amendment No. 1 to Northstar-Redwood Master Services Agreement (Fictional)","document_version":"Executed Amendment No. 1","as_of_date":"2024-09-15","page":4,"section_path":["6. Artificial Intelligence Use"],"content_type":"amendment","text":"Provider may not use Customer Data to train a general-purpose machine-learning model. Provider may use Customer Data for Customer-specific inference and retrieval only to deliver the services and subject to the Agreement's security terms.","parent_section_id":null,"previous_block_id":"AMD-007","next_block_id":"AMD-009","references":["MSA-024"]},{"block_id":"AMD-009","doc_id":"northstar_redwood_msa_amendment_1","document_title":"Amendment No. 1 to Northstar-Redwood Master Services Agreement (Fictional)","document_version":"Executed Amendment No. 1","as_of_date":"2024-09-15","page":5,"section_path":["7. Audit Reports"],"content_type":"amendment","text":"Once per calendar year, Provider will supply its then-current independent controls report on written request. Customer may conduct an additional audit only after a material confirmed security incident and subject to reasonable scope and confidentiality limits.","parent_section_id":null,"previous_block_id":"AMD-008","next_block_id":"AMD-010","references":["MSA-022","MSA-023"]},{"block_id":"AMD-010","doc_id":"northstar_redwood_msa_amendment_1","document_title":"Amendment No. 1 to Northstar-Redwood Master Services Agreement (Fictional)","document_version":"Executed Amendment No. 1","as_of_date":"2024-09-15","page":5,"section_path":["8. Ratification"],"content_type":"amendment","text":"The parties ratify the Agreement as modified by this Amendment. References to the Agreement after the effective date mean the Agreement as amended.","parent_section_id":null,"previous_block_id":"AMD-009","next_block_id":"AMD-011","references":["AMD-003","MSA-001"]},{"block_id":"AMD-011","doc_id":"northstar_redwood_msa_amendment_1","document_title":"Amendment No. 1 to Northstar-Redwood Master Services Agreement (Fictional)","document_version":"Executed Amendment No. 1","as_of_date":"2024-09-15","page":6,"section_path":["9. Counterparts"],"content_type":"clause","text":"This Amendment may be signed in counterparts and by electronic signature, each deemed an original and all constituting one instrument.","parent_section_id":null,"previous_block_id":"AMD-010","next_block_id":"AMD-012","references":[]},{"block_id":"AMD-012","doc_id":"northstar_redwood_msa_amendment_1","document_title":"Amendment No. 1 to Northstar-Redwood Master Services Agreement (Fictional)","document_version":"Executed Amendment No. 1","as_of_date":"2024-09-15","page":6,"section_path":["Signatures"],"content_type":"amendment","text":"Fictional signature blocks indicate execution by authorized representatives of Northstar Systems Group, Inc. and Redwood Harbor Bank, N.A. on September 15, 2024.","parent_section_id":null,"previous_block_id":"AMD-011","next_block_id":null,"references":["AMD-002"]}],"questions":[{"question_id":"Q-001","question":"What was Northstar's total revenue in 2024?","domain":"financial","question_type":"table_numeric","ground_truth":"Northstar's 2024 total revenue was $842.6 million.","is_answerable":true,"evidence_block_ids":["FIN-006"],"evidence_pages":[{"doc_id":"northstar_2024_annual_report","page":38}],"required_facts":["2024 revenue equals 842.6","Amounts are reported in USD millions"],"numeric_spec":{"expected":842.6,"unit":"USD millions","absolute_tolerance":0.05}},{"question_id":"Q-002","question":"By how much did Cloud Platform revenue increase from 2023 to 2024, in dollars and percent?","domain":"financial","question_type":"table_numeric","ground_truth":"Cloud Platform revenue increased by $74.3 million, approximately 17.0% (74.3 / 438.1).","is_answerable":true,"evidence_block_ids":["FIN-008"],"evidence_pages":[{"doc_id":"northstar_2024_annual_report","page":40}],"required_facts":["2024 Cloud Platform revenue is 512.4","2023 Cloud Platform revenue is 438.1","Absolute increase is 74.3","Percentage increase is approximately 16.96%"],"numeric_spec":{"expected":74.3,"unit":"USD millions","absolute_tolerance":0.05,"secondary_expected_percent":16.96,"percent_tolerance":0.1}},{"question_id":"Q-003","question":"Reconcile Northstar's 2024 free cash flow from the reported cash-flow components.","domain":"financial","question_type":"multi_hop","ground_truth":"Free cash flow was $97.4 million: $132.1 million of operating cash flow less $34.7 million of capital expenditures.","is_answerable":true,"evidence_block_ids":["FIN-018","FIN-019"],"evidence_pages":[{"doc_id":"northstar_2024_annual_report","page":50},{"doc_id":"northstar_2024_annual_report","page":51}],"required_facts":["Operating cash flow is 132.1","Capital expenditures are 34.7","Free cash flow definition subtracts capital expenditures","132.1 minus 34.7 equals 97.4"],"numeric_spec":{"expected":97.4,"unit":"USD millions","absolute_tolerance":0.05}},{"question_id":"Q-004","question":"How much debt principal is scheduled to mature in 2027 and 2028 combined?","domain":"financial","question_type":"table_numeric","ground_truth":"$140.0 million is scheduled to mature in 2027 and 2028 combined ($60.0 million + $80.0 million).","is_answerable":true,"evidence_block_ids":["FIN-024"],"evidence_pages":[{"doc_id":"northstar_2024_annual_report","page":58}],"required_facts":["2027 principal due is 60.0","2028 principal due is 80.0","Combined amount is 140.0"],"numeric_spec":{"expected":140.0,"unit":"USD millions","absolute_tolerance":0.05}},{"question_id":"Q-005","question":"Approximately how much of the year-end 2024 remaining performance obligations did Northstar expect to recognize in the next 12 months?","domain":"financial","question_type":"multi_hop","ground_truth":"Approximately $160.4 million, calculated as 61% of $263.0 million.","is_answerable":true,"evidence_block_ids":["FIN-011"],"evidence_pages":[{"doc_id":"northstar_2024_annual_report","page":42}],"required_facts":["Remaining performance obligations are 263.0","61% is expected within 12 months","263.0 multiplied by 0.61 equals 160.43"],"numeric_spec":{"expected":160.43,"unit":"USD millions","absolute_tolerance":0.1}},{"question_id":"Q-006","question":"What is the maximum reasonably possible Orion Claim loss above the amount already accrued?","domain":"financial","question_type":"multi_hop","ground_truth":"$1.5 million, equal to the $4.0 million top of the loss range less the $2.5 million accrual.","is_answerable":true,"evidence_block_ids":["FIN-037"],"evidence_pages":[{"doc_id":"northstar_2024_annual_report","page":101}],"required_facts":["Top of reasonably possible range is 4.0","Accrued amount is 2.5","Difference is 1.5"],"numeric_spec":{"expected":1.5,"unit":"USD millions","absolute_tolerance":0.05}},{"question_id":"Q-007","question":"What financial covenant thresholds applied at year end, and was Northstar compliant?","domain":"financial","question_type":"multi_hop","ground_truth":"The net leverage ratio could not exceed 3.50:1 and interest coverage could not fall below 3.00:1. Actual ratios were 1.42:1 and 8.10:1, respectively, so Northstar was compliant.","is_answerable":true,"evidence_block_ids":["FIN-022","FIN-023"],"evidence_pages":[{"doc_id":"northstar_2024_annual_report","page":56}],"required_facts":["Net leverage maximum is 3.50:1","Interest coverage minimum is 3.00:1","Actual net leverage is 1.42:1","Actual interest coverage is 8.10:1","Company states it was compliant"]},{"question_id":"Q-008","question":"After Amendment No. 1, what is the deadline for notifying Redwood of a confirmed unauthorized access incident?","domain":"legal","question_type":"amendment_conflict","ground_truth":"No later than 24 hours after confirmation. Amendment No. 1 replaces the MSA's original 48-hour deadline.","is_answerable":true,"evidence_block_ids":["MSA-023","AMD-003","AMD-004"],"evidence_pages":[{"doc_id":"northstar_redwood_msa","page":15},{"doc_id":"northstar_redwood_msa_amendment_1","page":2}],"required_facts":["Original deadline is 48 hours after confirmation","Amendment controls in a conflict","Amended deadline is 24 hours after confirmation"]},{"question_id":"Q-009","question":"What liability caps apply after Amendment No. 1, including the uncapped categories?","domain":"legal","question_type":"amendment_conflict","ground_truth":"The amended general cap is 1.5 times the relevant prior-12-month Fees; confidentiality and security claims have a 3-times cap; fraud, willful misconduct, and Provider's IP indemnity are uncapped.","is_answerable":true,"evidence_block_ids":["MSA-029","AMD-003","AMD-005"],"evidence_pages":[{"doc_id":"northstar_redwood_msa","page":23},{"doc_id":"northstar_redwood_msa_amendment_1","page":2},{"doc_id":"northstar_redwood_msa_amendment_1","page":3}],"required_facts":["Original general cap was 1 times Fees","Amendment replaces Section 10","Amended general cap is 1.5 times Fees","Amended confidentiality/security cap is 3 times Fees","Fraud, willful misconduct, and Provider IP indemnity are uncapped"]},{"question_id":"Q-010","question":"How much notice must Redwood give to terminate for convenience during the initial term versus a later renewal period?","domain":"legal","question_type":"amendment_conflict","ground_truth":"The initial term retains the original 90-day notice period. A renewal period beginning after the amendment effective date uses 60 days' prior written notice.","is_answerable":true,"evidence_block_ids":["MSA-015","MSA-016","AMD-007"],"evidence_pages":[{"doc_id":"northstar_redwood_msa","page":10},{"doc_id":"northstar_redwood_msa_amendment_1","page":4}],"required_facts":["Original convenience notice is 90 days","Initial term remains subject to 90 days","Qualifying renewal periods use 60 days"]},{"question_id":"Q-011","question":"How long do the MSA confidentiality duties last, and how are trade secrets treated?","domain":"legal","question_type":"direct","ground_truth":"The duties last five years after disclosure, while qualifying trade secrets remain protected for as long as they retain trade-secret status under applicable law.","is_answerable":true,"evidence_block_ids":["MSA-019"],"evidence_pages":[{"doc_id":"northstar_redwood_msa","page":12}],"required_facts":["General confidentiality duration is five years after disclosure","Trade secrets remain protected while they qualify as trade secrets"]},{"question_id":"Q-012","question":"When is a deliverable deemed accepted under the MSA?","domain":"legal","question_type":"direct","ground_truth":"It is deemed accepted if Redwood does not provide written notice of material nonconformities within ten business days after delivery.","is_answerable":true,"evidence_block_ids":["MSA-010"],"evidence_pages":[{"doc_id":"northstar_redwood_msa","page":7}],"required_facts":["Review period is ten business days after delivery","Notice must be written and describe material nonconformities","No timely notice results in deemed acceptance"]},{"question_id":"Q-013","question":"How should a conflict among Amendment No. 1, the MSA, and a statement of work be resolved?","domain":"legal","question_type":"cross_reference","ground_truth":"Amendment No. 1 controls over both. Otherwise, a statement of work controls its specific business terms, while the MSA controls legal terms unless the statement expressly overrides an identified MSA section.","is_answerable":true,"evidence_block_ids":["MSA-008","AMD-003"],"evidence_pages":[{"doc_id":"northstar_redwood_msa","page":5},{"doc_id":"northstar_redwood_msa_amendment_1","page":2}],"required_facts":["Amendment controls conflicts","SOW controls specific business terms","MSA controls legal terms absent an express section override"]},{"question_id":"Q-014","question":"Who are the parties, and when did the original MSA and Amendment No. 1 become effective?","domain":"legal","question_type":"cross_reference","ground_truth":"The parties are Northstar Systems Group, Inc. (Provider) and Redwood Harbor Bank, N.A. (Customer). The MSA became effective April 1, 2023; Amendment No. 1 became effective September 15, 2024.","is_answerable":true,"evidence_block_ids":["MSA-002","AMD-002"],"evidence_pages":[{"doc_id":"northstar_redwood_msa","page":1},{"doc_id":"northstar_redwood_msa_amendment_1","page":1}],"required_facts":["Provider is Northstar Systems Group, Inc.","Customer is Redwood Harbor Bank, N.A.","MSA effective date is 2023-04-01","Amendment effective date is 2024-09-15"]},{"question_id":"Q-015","question":"Identify the report pages that provide 2024 total revenue, the debt-maturity schedule, and the 2024 effective tax rate, and state each value.","domain":"financial","question_type":"global","ground_truth":"Total revenue of $842.6 million appears on page 38; the debt-maturity schedule appears on page 58 (total principal $165.0 million); and the 23.8% effective tax rate appears on page 91.","is_answerable":true,"evidence_block_ids":["FIN-006","FIN-024","FIN-033"],"evidence_pages":[{"doc_id":"northstar_2024_annual_report","page":38},{"doc_id":"northstar_2024_annual_report","page":58},{"doc_id":"northstar_2024_annual_report","page":91}],"required_facts":["Page 38 reports 842.6 total revenue","Page 58 reports 165.0 total debt principal across maturity years","Page 91 reports a 23.8% effective tax rate"]},{"question_id":"Q-016","question":"What numerical revenue guidance did Northstar issue for fiscal 2025?","domain":"financial","question_type":"unanswerable","ground_truth":"Not answerable from the benchmark documents; no fiscal 2025 numerical revenue guidance is provided.","is_answerable":false,"evidence_block_ids":[],"evidence_pages":[],"required_facts":["Fiscal 2025 numerical revenue guidance"]},{"question_id":"Q-017","question":"What dollar limit of cyber-liability insurance is Northstar required to maintain under the MSA?","domain":"legal","question_type":"unanswerable","ground_truth":"Not answerable from the benchmark documents; no insurance coverage amount or cyber-liability policy limit is stated.","is_answerable":false,"evidence_block_ids":[],"evidence_pages":[],"required_facts":["Required cyber-liability insurance limit"]},{"question_id":"Q-018","question":"How much did Northstar pay Lantern & Moss LLP in audit fees for 2024?","domain":"financial","question_type":"unanswerable","ground_truth":"Not answerable from the benchmark documents; the fictional auditor is named, but audit fees are not disclosed.","is_answerable":false,"evidence_block_ids":[],"evidence_pages":[],"required_facts":["2024 audit fees paid to Lantern & Moss LLP"]}]}"""

def load_large_document_benchmark():
    """Load the repository fixture, with a complete embedded fallback for standalone Colab."""
    candidates = [
        Path("../data/large_document_benchmark.json"),
        Path("data/large_document_benchmark.json"),
        Path("/content/agentic-ops-and-rag/data/large_document_benchmark.json"),
    ]
    for candidate in candidates:
        if candidate.exists():
            with candidate.open(encoding="utf-8") as handle:
                return json.load(handle), str(candidate)
    return json.loads(EMBEDDED_BENCHMARK_JSON), "embedded fallback"

benchmark, benchmark_source = load_large_document_benchmark()
blocks = benchmark["blocks"]
questions = benchmark["questions"]
block_by_id = {block["block_id"]: block for block in blocks}

print(f"Loaded {len(blocks)} provenance-preserving blocks and {len(questions)} questions from {benchmark_source}")
print("Documents:")
for doc_id in sorted({block['doc_id'] for block in blocks}):
    doc_blocks = [block for block in blocks if block["doc_id"] == doc_id]
    print(f"  {doc_id}: {len(doc_blocks)} blocks, pages {min(b['page'] for b in doc_blocks)}-{max(b['page'] for b in doc_blocks)}")


<h2 id="schema">7.1 — Evidence-first gold schema</h2>

Every case supplies question type, answerability, minimal evidence block IDs, pages,
required facts, and optional typed numeric expectations. Production annotations should
also include character spans or table cells/bounding boxes and OR-of-AND acceptable
evidence sets. Split by document/company/contract—not random question rows—to prevent
near-duplicate leakage.


In [ ]:
# Validate referential integrity before trusting any score.
errors = []
for question in questions:
    for evidence_id in question["evidence_block_ids"]:
        if evidence_id not in block_by_id:
            errors.append(f"{question['question_id']}: missing {evidence_id}")
    actual_pages = {(block_by_id[eid]["doc_id"], block_by_id[eid]["page"]) for eid in question["evidence_block_ids"]}
    labeled_pages = {(item["doc_id"], item["page"]) for item in question["evidence_pages"]}
    if actual_pages != labeled_pages:
        errors.append(f"{question['question_id']}: page labels do not match evidence")
assert not errors, errors
print("PASS: all evidence IDs and page-level provenance labels resolve.")
print(json.dumps(questions[8], indent=2))


In [ ]:
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "did", "do", "does", "for",
    "from", "how", "in", "is", "it", "of", "on", "or", "the", "their", "there", "to",
    "under", "was", "were", "what", "when", "which", "who", "with", "would",
}

CONCEPT_ALIASES = {
    "sales": "revenue", "turnover": "revenue", "earnings": "income",
    "borrowings": "debt", "borrowing": "debt", "maturity": "mature",
    "maturities": "mature", "covenants": "covenant", "ratios": "ratio",
    "agreement": "msa", "contract": "msa", "clause": "section",
    "provider": "northstar", "customer": "redwood", "bank": "redwood",
    "breach": "violation", "confidential": "confidentiality",
    "notification": "notice", "notify": "notice", "notifying": "notice",
    "caps": "cap", "capped": "cap", "limitation": "cap", "liability": "cap",
    "terminate": "termination", "terminated": "termination", "renewal": "term",
    "amended": "amendment", "amends": "amendment", "modified": "amendment",
    "effective": "date", "became": "date", "deadline": "hours",
}

def tokenize(text):
    return re.findall(r"[a-z0-9]+(?:\.[0-9]+)?", text.lower())

def concept_tokens(text):
    terms = []
    for token in tokenize(text):
        if token in STOPWORDS:
            continue
        token = CONCEPT_ALIASES.get(token, token)
        if len(token) > 4 and token.endswith("s") and not token.endswith("ss"):
            token = token[:-1]
        terms.append(token)
    return terms

def section_id(block):
    root = block["section_path"][0] if block["section_path"] else "Unsectioned"
    return f"{block['doc_id']}::{root}"

def contextualize_block(block):
    """Index this representation; always return original `text` as cited evidence."""
    path = " > ".join(block["section_path"])
    return (
        f"Document: {block['document_title']}\n"
        f"Version/date: {block['document_version']} / {block['as_of_date']}\n"
        f"Page {block['page']} | Section: {path} | Type: {block['content_type']}\n"
        f"{block['text']}"
    )

class BM25:
    """Small transparent BM25 implementation; production systems use an inverted index."""
    def __init__(self, texts):
        self.docs = [concept_tokens(text) for text in texts]
        self.avgdl = sum(map(len, self.docs)) / max(1, len(self.docs))
        self.df = Counter()
        for doc in self.docs:
            self.df.update(set(doc))
        self.n = len(self.docs)

    def score(self, query, k1=1.5, b=0.75):
        qterms = concept_tokens(query)
        scores = []
        for doc in self.docs:
            tf = Counter(doc)
            value = 0.0
            for term in qterms:
                freq = tf[term]
                if not freq:
                    continue
                idf = math.log(1.0 + (self.n - self.df[term] + 0.5) / (self.df[term] + 0.5))
                denom = freq + k1 * (1 - b + b * len(doc) / max(1.0, self.avgdl))
                value += idf * freq * (k1 + 1) / denom
            scores.append(value)
        return scores

def conceptual_similarity(query, text):
    q = Counter(concept_tokens(query))
    d = Counter(concept_tokens(text))
    if not q or not d:
        return 0.0
    dot = sum(q[t] * d[t] for t in q)
    return dot / math.sqrt(sum(v*v for v in q.values()) * sum(v*v for v in d.values()))

def route_query(query):
    q = query.lower()
    legal_markers = {"msa", "amendment", "redwood", "clause", "liability", "confidentiality",
                     "deliverable", "provider", "customer", "terminate", "agreement"}
    domain = "legal" if any(marker in q for marker in legal_markers) else "financial"
    if any(marker in q for marker in ("after amendment", "amendment no", "versus a later", "as amended")):
        mode = "amendment_conflict"
    elif any(marker in q for marker in ("conflict among", "who are the parties", "cross-reference", "pursuant")):
        mode = "cross_reference"
    elif any(marker in q for marker in ("identify the report pages", "across the report", "summarize", "overall")):
        mode = "global"
    elif any(marker in q for marker in ("how much", "percent", "ratio", "reconcile", "combined", "total revenue")):
        mode = "table_numeric"
    else:
        mode = "lookup"
    preferred_types = {"table", "footnote"} if mode == "table_numeric" else set()
    return {
        "domain": domain,
        "mode": mode,
        "preferred_types": preferred_types,
        "expand_references": mode in {"amendment_conflict", "cross_reference", "table_numeric"},
    }

def decompose_global_query(query):
    """Extract independently retrievable targets from a broad compound question."""
    q = query.lower()
    year_match = re.search(r"\b(20\d{2})\b", q)
    year = year_match.group(1) if year_match else ""
    # Filing-aware mappings are a transparent stand-in for a learned query planner.
    # They encode where standardized reports normally place a requested fact.
    mapped_targets = []
    if "total revenue" in q:
        mapped_targets.append(f"{year} revenue consolidated statements of operations".strip())
    if "debt" in q and "matur" in q:
        mapped_targets.append("debt maturity schedule principal due")
    if "effective tax rate" in q:
        mapped_targets.append(f"{year} effective tax rate income taxes".strip())
    if mapped_targets:
        return mapped_targets
    phrase_patterns = [
        r"(?:\d{4}\s+)?total revenue",
        r"debt[- ]matur(?:ity|ities|e)[a-z ]*schedule",
        r"(?:\d{4}\s+)?effective tax rate",
        r"cash[- ]flow components?",
        r"financial covenant thresholds?",
    ]
    targets = []
    for pattern in phrase_patterns:
        match = re.search(pattern, q)
        if match:
            targets.append(match.group(0).strip())
    if targets:
        return list(dict.fromkeys(targets))
    clauses = [part.strip(" .?") for part in re.split(r",|\band\b", q) if len(concept_tokens(part)) >= 2]
    return clauses if len(clauses) > 1 else [query]

class LargeDocumentIndex:
    def __init__(self, blocks):
        self.blocks = list(blocks)
        self.by_id = {b["block_id"]: b for b in self.blocks}
        self.contexts = [contextualize_block(b) for b in self.blocks]
        self.raw_bm25 = BM25([b["text"] for b in self.blocks])
        self.context_bm25 = BM25(self.contexts)
        self.section_members = defaultdict(list)
        for block in self.blocks:
            self.section_members[section_id(block)].append(block)
        self.section_ids = sorted(self.section_members)
        self.section_summaries = []
        for sid in self.section_ids:
            members = self.section_members[sid]
            headings = "; ".join(dict.fromkeys(" > ".join(b["section_path"]) for b in members))
            preview = " ".join(b["text"][:180] for b in members)
            self.section_summaries.append(f"{members[0]['document_title']} | {headings} | {preview}")
        self.section_bm25 = BM25(self.section_summaries)

    def shortlist_sections(self, query, top_m=4, domain=None):
        scores = self.section_bm25.score(query)
        ranked = []
        for sid, score in zip(self.section_ids, scores):
            first = self.section_members[sid][0]
            is_legal = first["doc_id"].startswith("northstar_redwood")
            if domain == "legal" and not is_legal:
                continue
            if domain == "financial" and is_legal:
                continue
            ranked.append((sid, score))
        return [sid for sid, _ in sorted(ranked, key=lambda pair: (-pair[1], pair[0]))[:top_m]]

    def score_blocks(self, query, *, contextual=True, hybrid=True, section_filter=None, domain=None):
        lexical = self.context_bm25.score(query) if contextual else self.raw_bm25.score(query)
        max_lexical = max(lexical) or 1.0
        route = route_query(query)
        ranked = []
        for idx, block in enumerate(self.blocks):
            is_legal = block["doc_id"].startswith("northstar_redwood")
            if domain == "legal" and not is_legal:
                continue
            if domain == "financial" and is_legal:
                continue
            if section_filter and section_id(block) not in section_filter:
                continue
            lexical_norm = lexical[idx] / max_lexical
            semantic = conceptual_similarity(query, self.contexts[idx]) if hybrid else 0.0
            heading = conceptual_similarity(query, " ".join(block["section_path"]))
            score = (0.68 * lexical_norm + 0.22 * semantic + 0.10 * heading) if hybrid else lexical_norm
            if block["content_type"] in route["preferred_types"]:
                score += 0.08
            if route["mode"] == "amendment_conflict" and block["doc_id"].endswith("amendment_1"):
                score += 0.12
            ranked.append({"block_id": block["block_id"], "score": score, "reason": "retrieved"})
        return sorted(ranked, key=lambda item: (-item["score"], item["block_id"]))

    def retrieve(self, query, *, top_k=6, contextual=True, hybrid=True,
                 hierarchical=False, expand=False, section_top_m=4):
        route = route_query(query)
        if hierarchical and route["mode"] == "global":
            merged = {}
            for subquery in decompose_global_query(query):
                sub_sections = self.shortlist_sections(subquery, top_m=2, domain=route["domain"])
                sub_ranked = self.score_blocks(
                    subquery, contextual=contextual, hybrid=hybrid,
                    section_filter=sub_sections, domain=route["domain"],
                )[:2]
                for rank, item in enumerate(sub_ranked, 1):
                    candidate = dict(item)
                    candidate["score"] += 0.12 / rank
                    candidate["reason"] = f"subquery: {subquery}"
                    old = merged.get(candidate["block_id"])
                    if old is None or candidate["score"] > old["score"]:
                        merged[candidate["block_id"]] = candidate
            ranked = sorted(merged.values(), key=lambda item: (-item["score"], item["block_id"]))[:top_k]
            return self._expand(query, ranked, top_k=top_k) if expand else ranked
        sections = None
        # Global questions may need several branches; local questions use a narrower shortlist.
        if hierarchical:
            parent_count = 6 if route["mode"] == "global" else section_top_m
            sections = self.shortlist_sections(query, top_m=parent_count, domain=route["domain"])
        ranked = self.score_blocks(
            query, contextual=contextual, hybrid=hybrid,
            section_filter=sections, domain=route["domain"],
        )
        selected = ranked[:top_k]
        if expand:
            selected = self._expand(query, selected, top_k=top_k)
        return selected

    def _expand(self, query, selected, top_k):
        """Follow explicit references and same-section neighbors, then deduplicate and rerank."""
        route = route_query(query)
        merged = {item["block_id"]: dict(item) for item in selected}
        seeds = list(selected[: max(2, min(4, len(selected)))])
        for item in seeds:
            block = self.by_id[item["block_id"]]
            if route["expand_references"] or block["doc_id"].endswith("amendment_1"):
                for ref_id in block["references"]:
                    if ref_id in self.by_id:
                        candidate = {"block_id": ref_id, "score": item["score"] * 0.94, "reason": f"reference from {block['block_id']}"}
                        if ref_id not in merged or candidate["score"] > merged[ref_id]["score"]:
                            merged[ref_id] = candidate
            for neighbor_id in (block["previous_block_id"], block["next_block_id"]):
                if not neighbor_id or neighbor_id not in self.by_id:
                    continue
                neighbor = self.by_id[neighbor_id]
                if section_id(neighbor) == section_id(block):
                    candidate = {"block_id": neighbor_id, "score": item["score"] * 0.72, "reason": f"neighbor of {block['block_id']}"}
                    if neighbor_id not in merged:
                        merged[neighbor_id] = candidate
        reranked = []
        for item in merged.values():
            block = self.by_id[item["block_id"]]
            item["score"] += 0.08 * conceptual_similarity(query, contextualize_block(block))
            reranked.append(item)
        # Expansion is allowed to add evidence beyond top_k, but is deliberately bounded.
        return sorted(reranked, key=lambda item: (-item["score"], item["block_id"]))[: top_k + 4]

def citation_for(block):
    section = " > ".join(block["section_path"])
    return f"[{block['doc_id']} p.{block['page']} § {section}]"

def estimate_tokens(text):
    # A deterministic proxy suitable for this lab; use the serving model tokenizer in production.
    return max(1, math.ceil(len(re.findall(r"\S+", text)) * 1.25))

def pack_context(candidates, by_id, token_budget=420):
    """Greedily pack original evidence while preserving citations and avoiding near-duplicates."""
    packed, used_tokens, seen_terms = [], 0, []
    for candidate in candidates:
        block = by_id[candidate["block_id"]]
        payload = f"{citation_for(block)}\n{block['text']}"
        cost = estimate_tokens(payload)
        terms = set(concept_tokens(block["text"]))
        redundancy = max((len(terms & prior) / max(1, len(terms | prior)) for prior in seen_terms), default=0.0)
        if redundancy > 0.82 or used_tokens + cost > token_budget:
            continue
        packed.append({**candidate, "citation": citation_for(block), "text": block["text"], "tokens": cost})
        used_tokens += cost
        seen_terms.append(terms)
    return {"items": packed, "tokens": used_tokens, "budget": token_budget}


In [ ]:
def rank_ids(index, question, config):
    route = route_query(question["question"])
    candidates = index.retrieve(
        question["question"],
        top_k=config.get("top_k", 6),
        contextual=config.get("contextual", False),
        hybrid=config.get("hybrid", False),
        hierarchical=config.get("hierarchical", False),
        expand=config.get("expand", False),
        section_top_m=config.get("section_top_m", 4),
    )
    packet = pack_context(candidates, index.by_id, token_budget=config.get("token_budget", 360))
    return candidates, packet

def evidence_recall_at_k(gold_ids, ranked_ids, k):
    gold = set(gold_ids)
    if not gold:
        return 1.0
    return len(gold & set(ranked_ids[:k])) / len(gold)

def strict_evidence_recall_at_k(gold_ids, ranked_ids, k):
    gold = set(gold_ids)
    return float(gold.issubset(set(ranked_ids[:k]))) if gold else 1.0

def reciprocal_rank(gold_ids, ranked_ids):
    gold = set(gold_ids)
    for rank, block_id in enumerate(ranked_ids, 1):
        if block_id in gold:
            return 1.0 / rank
    return 0.0

def ndcg_at_k(gold_ids, ranked_ids, k):
    gold = set(gold_ids)
    if not gold:
        return 1.0
    dcg = sum((1.0 / math.log2(rank + 1)) for rank, bid in enumerate(ranked_ids[:k], 1) if bid in gold)
    ideal = sum(1.0 / math.log2(rank + 1) for rank in range(1, min(k, len(gold)) + 1))
    return dcg / ideal if ideal else 0.0

def evaluate_retriever(index, questions, config):
    rows = []
    for question in questions:
        candidates, packet = rank_ids(index, question, config)
        candidate_ids = [item["block_id"] for item in candidates]
        packed_ids = [item["block_id"] for item in packet["items"]]
        gold = question["evidence_block_ids"]
        useful_tokens = sum(item["tokens"] for item in packet["items"] if item["block_id"] in set(gold))
        rows.append({
            "question_id": question["question_id"],
            "question_type": question["question_type"],
            "answerable": question["is_answerable"],
            "candidate_ids": candidate_ids,
            "packed_ids": packed_ids,
            "recall": evidence_recall_at_k(gold, packed_ids, len(packed_ids)),
            "strict_recall": strict_evidence_recall_at_k(gold, packed_ids, len(packed_ids)),
            "mrr": reciprocal_rank(gold, candidate_ids) if gold else 1.0,
            "ndcg": ndcg_at_k(gold, candidate_ids, max(1, len(candidate_ids))),
            "context_precision": len(set(gold) & set(packed_ids)) / max(1, len(packed_ids)),
            "tokens": packet["tokens"],
            "evidence_density": useful_tokens / max(1, packet["tokens"]),
        })
    return rows

def summarize_rows(rows):
    answerable = [row for row in rows if row["answerable"]]
    return {
        "evidence_recall": statistics.mean(row["recall"] for row in answerable),
        "strict_recall": statistics.mean(row["strict_recall"] for row in answerable),
        "mrr": statistics.mean(row["mrr"] for row in answerable),
        "ndcg": statistics.mean(row["ndcg"] for row in answerable),
        "context_precision": statistics.mean(row["context_precision"] for row in answerable),
        "evidence_density": statistics.mean(row["evidence_density"] for row in answerable),
        "mean_tokens": statistics.mean(row["tokens"] for row in rows),
    }

def first_failure(question, candidates, packet, reader_correct=True, citations_complete=True):
    gold = set(question["evidence_block_ids"])
    if not question["is_answerable"]:
        return "unsafe_answer" if reader_correct is False else "none"
    if not gold.issubset(index.by_id):
        return "parse_or_source_map_failed"
    candidate_ids = {item["block_id"] for item in candidates}
    if not gold.issubset(candidate_ids):
        return "retrieval_failed"
    packed_ids = {item["block_id"] for item in packet["items"]}
    if not gold.issubset(packed_ids):
        return "packing_truncated_evidence"
    if not reader_correct:
        return "reader_or_reasoning_failed"
    if not citations_complete:
        return "citation_failed"
    return "none"

def paired_bootstrap_delta(baseline, challenger, iterations=2000, seed=7):
    if len(baseline) != len(challenger) or not baseline:
        raise ValueError("Paired non-empty samples of equal length are required")
    rng = random.Random(seed)
    deltas = []
    n = len(baseline)
    for _ in range(iterations):
        indices = [rng.randrange(n) for _ in range(n)]
        deltas.append(statistics.mean(challenger[i] - baseline[i] for i in indices))
    deltas.sort()
    lo = deltas[int(0.025 * iterations)]
    hi = deltas[min(iterations - 1, int(0.975 * iterations))]
    observed = statistics.mean(c - b for b, c in zip(baseline, challenger))
    return {"delta": observed, "ci95": (lo, hi)}


<h2 id="metrics">7.2 — Retrieval metrics that diagnose multi-hop misses</h2>

For a two-block answer, ordinary recall gives partial credit when one fact is found;
**strict evidence recall** stays zero until every required source is present. MRR measures
the first relevant hit, while nDCG rewards putting all relevant evidence near the top.
Context precision and evidence-token density then expose “recall by prompt dumping.”


In [ ]:
def student_retrieval_metrics(gold_ids, ranked_ids, k):
    """Return recall@k, strict all-evidence recall@k, MRR, and nDCG@k."""
    return {
        "recall": evidence_recall_at_k(gold_ids, ranked_ids, k),
        "strict_recall": strict_evidence_recall_at_k(gold_ids, ranked_ids, k),
        "mrr": reciprocal_rank(gold_ids, ranked_ids),
        "ndcg": ndcg_at_k(gold_ids, ranked_ids, k),
    }


In [ ]:
# Verification — do not modify.
gold = ["A", "C"]
ranked = ["X", "A", "Y", "C"]
m = student_retrieval_metrics(gold, ranked, k=3)
assert m["recall"] == 0.5
assert m["strict_recall"] == 0.0
assert abs(m["mrr"] - 0.5) < 1e-9
assert 0.0 < m["ndcg"] < 1.0
assert student_retrieval_metrics(gold, ranked, k=4)["strict_recall"] == 1.0
print("PASS:", m)


<h2 id="ablation">7.3 — Controlled 2×2×2 factorial experiment</h2>

Hold the corpus, question set, token budget, and top-k fixed. Vary only:

- raw vs contextualized block representation;
- lexical-only vs hybrid conceptual matching;
- direct top-k vs reference/neighbor expansion.

This reveals interactions. For example, expansion may improve strict recall but reduce
precision, while contextualization may help ambiguous amendment fragments.


In [ ]:
index = LargeDocumentIndex(blocks)
experiment_rows = []
experiment_outputs = {}
for contextual in (False, True):
    for hybrid in (False, True):
        for expand in (False, True):
            name = f"ctx={int(contextual)}|hybrid={int(hybrid)}|expand={int(expand)}"
            config = {
                "contextual": contextual, "hybrid": hybrid, "expand": expand,
                "hierarchical": False, "top_k": 6, "token_budget": 360,
            }
            start = time.perf_counter()
            rows = evaluate_retriever(index, questions, config)
            elapsed_ms = 1000 * (time.perf_counter() - start)
            summary = summarize_rows(rows)
            summary.update({"name": name, "latency_ms": elapsed_ms})
            experiment_rows.append(summary)
            experiment_outputs[name] = rows

headers = ["name", "evidence_recall", "strict_recall", "mrr", "context_precision", "evidence_density", "mean_tokens", "latency_ms"]
print(" | ".join(f"{h:>17s}" for h in headers))
for row in sorted(experiment_rows, key=lambda item: (-item["strict_recall"], -item["evidence_recall"])):
    print(" | ".join(f"{row[h]:17.3f}" if isinstance(row[h], float) else f"{str(row[h]):>17s}" for h in headers))


<h2 id="hierarchical">7.4 — Add the large-document architecture, then inspect slices</h2>

The factorial grid isolates basic retrieval choices. Now compare the flat baseline with
the complete hierarchy + expansion path from Module 6. Never accept a macro win until
table/numeric, multi-hop, amendment, cross-reference, global, and unanswerable slices are
visible separately.


In [ ]:
BASELINE = {"contextual": False, "hybrid": False, "hierarchical": False,
            "expand": False, "top_k": 5, "token_budget": 360}
ADVANCED = {"contextual": True, "hybrid": True, "hierarchical": True,
            "expand": True, "top_k": 7, "section_top_m": 5, "token_budget": 420}

baseline_rows = evaluate_retriever(index, questions, BASELINE)
advanced_rows = evaluate_retriever(index, questions, ADVANCED)
baseline_summary = summarize_rows(baseline_rows)
advanced_summary = summarize_rows(advanced_rows)
print("Baseline:", {k: round(v, 3) for k, v in baseline_summary.items()})
print("Advanced:", {k: round(v, 3) for k, v in advanced_summary.items()})

print("\nStrict evidence recall by slice")
for question_type in sorted({q["question_type"] for q in questions if q["is_answerable"]}):
    b = [r["strict_recall"] for r in baseline_rows if r["answerable"] and r["question_type"] == question_type]
    a = [r["strict_recall"] for r in advanced_rows if r["answerable"] and r["question_type"] == question_type]
    print(f"  {question_type:20s} baseline={statistics.mean(b):.3f} advanced={statistics.mean(a):.3f} n={len(a)}")


### Earliest-failure attribution

Run candidates and packed context separately. If gold appeared in candidates but vanished
from the packet, fix the packer—not the embedding model. Use oracle evidence → reader runs
to measure the generation/reasoning ceiling independently.


In [ ]:
failures = Counter()
for question in questions:
    candidates, packet = rank_ids(index, question, ADVANCED)
    failures[first_failure(question, candidates, packet)] += 1
print("First-failure waterfall:")
for failure, count in failures.most_common():
    print(f"  {failure:30s} {count}")


<h2 id="citation">7.5 — Score answers, citations, numbers, and abstention separately</h2>

Citation precision asks whether cited sources support the claims; citation completeness asks
whether every verifiable claim is cited. Numeric correctness additionally requires value,
unit/scale, sign, entity, and reporting period—not merely a nearby number.

The deterministic reader below is an **instrumented proxy**, not an LLM benchmark. It lets
us test metric plumbing and causally isolate retrieval: a correct answer is possible only
when every gold evidence block survives packing.


In [ ]:
def simulate_reader(question, packed_ids, abstain_on_missing=True):
    gold = set(question["evidence_block_ids"])
    present = gold & set(packed_ids)
    if not question["is_answerable"]:
        return {"action": "abstain", "answer_correct": True, "citations": []}
    if gold.issubset(present):
        return {"action": "answer", "answer_correct": True, "citations": sorted(gold)}
    if abstain_on_missing:
        return {"action": "abstain", "answer_correct": False, "citations": sorted(present)}
    return {"action": "answer", "answer_correct": False, "citations": sorted(present)}

def citation_scores(question, citation_ids):
    gold, cited = set(question["evidence_block_ids"]), set(citation_ids)
    return {
        "precision": len(gold & cited) / max(1, len(cited)),
        "completeness": len(gold & cited) / max(1, len(gold)),
    }

answer_rows = []
for question, retrieval in zip(questions, advanced_rows):
    result = simulate_reader(question, retrieval["packed_ids"])
    cites = citation_scores(question, result["citations"])
    answer_rows.append({"qid": question["question_id"], **result, **cites})

print("Answer accuracy:", statistics.mean(row["answer_correct"] for row in answer_rows))
print("Citation precision:", statistics.mean(row["precision"] for row in answer_rows if row["action"] == "answer"))
print("Citation completeness:", statistics.mean(row["completeness"] for row in answer_rows if row["action"] == "answer"))
assert all(row["action"] == "abstain" for row, q in zip(answer_rows, questions) if not q["is_answerable"])
print("Unsafe false-answer rate on unanswerables: 0.000")


In [ ]:
def normalized_numeric_score(predicted_value, predicted_unit, question):
    spec = question.get("numeric_spec")
    if not spec:
        return None
    value_ok = abs(float(predicted_value) - float(spec["expected"])) <= spec["absolute_tolerance"]
    unit_ok = predicted_unit.strip().lower() == spec["unit"].strip().lower()
    return {"value_correct": value_ok, "unit_correct": unit_ok, "fully_correct": value_ok and unit_ok}

print(normalized_numeric_score(842.6, "USD millions", questions[0]))
print(normalized_numeric_score(842.6, "USD", questions[0]), "# right number, wrong scale")


<h2 id="stress">7.6 — Position, corruption, and adversarial stress tests</h2>

At minimum, stratify by beginning/middle/end, evidence distance, modality, hops, and version.
Create answer-preserving and answer-changing mutations with explicit expected behavior:

- same metric/wrong company or fiscal period;
- superseded clause vs controlling amendment;
- percent vs percentage points and dollars vs millions;
- swapped table headers, OCR digit corruption, missing footnote;
- plausible-but-absent answer, false premise, and embedded prompt injection.

The test below corrupts only indexed text (source labels remain immutable), making the
quality loss measurable without silently changing the gold set.


In [ ]:
def corrupt_text(text, probability=0.08, seed=11):
    rng = random.Random(seed)
    substitutions = {"0": "O", "1": "l", "5": "S", "8": "B",
                     "a": "o", "e": "c", "i": "l", "o": "0", "s": "5"}
    return "".join(substitutions.get(char, char) if rng.random() < probability else char for char in text)

corrupted_blocks = []
for i, block in enumerate(blocks):
    clone = dict(block)
    clone["text"] = corrupt_text(block["text"], probability=0.22, seed=100 + i)
    corrupted_blocks.append(clone)
corrupted_index = LargeDocumentIndex(corrupted_blocks)
clean = summarize_rows(evaluate_retriever(index, questions, ADVANCED))
corrupt = summarize_rows(evaluate_retriever(corrupted_index, questions, ADVANCED))
print(f"Clean strict recall:     {clean['strict_recall']:.3f}")
print(f"OCR-damaged strict recall: {corrupt['strict_recall']:.3f}")

def evidence_position(block):
    max_page = max(b["page"] for b in blocks if b["doc_id"] == block["doc_id"])
    ratio = block["page"] / max_page
    return "beginning" if ratio <= 1/3 else "middle" if ratio <= 2/3 else "end"

print("\nAdvanced strict recall by first-evidence position:")
for position in ("beginning", "middle", "end"):
    values = []
    for question, row in zip(questions, advanced_rows):
        if question["is_answerable"] and evidence_position(block_by_id[question["evidence_block_ids"][0]]) == position:
            values.append(row["strict_recall"])
    if values:
        print(f"  {position:10s}: {statistics.mean(values):.3f} (n={len(values)})")


<h2 id="uncertainty">7.7 — Paired uncertainty, Pareto trade-offs, and a release gate</h2>

Queries are the paired unit: both systems face the same case. Bootstrap paired indices,
not two independent samples. For binary critical errors, add McNemar's test; when comparing
many variants, correct for multiplicity. Report worst-cohort and production-weighted scores
alongside the macro mean.


In [ ]:
def student_paired_bootstrap(baseline, challenger, iterations=2000, seed=7):
    """Estimate the paired mean delta and percentile 95% interval."""
    return paired_bootstrap_delta(baseline, challenger, iterations=iterations, seed=seed)


In [ ]:
# Verification — do not modify.
demo = student_paired_bootstrap([0, 0, 1, 0], [1, 0, 1, 1], iterations=1000, seed=3)
assert abs(demo["delta"] - 0.5) < 1e-9
assert demo["ci95"][0] <= demo["delta"] <= demo["ci95"][1]
print("PASS:", demo)

b = [row["strict_recall"] for row in baseline_rows if row["answerable"]]
a = [row["strict_recall"] for row in advanced_rows if row["answerable"]]
print("Observed advanced-baseline strict-recall delta:", student_paired_bootstrap(b, a, seed=17))


In [ ]:
# Quality/latency/token Pareto candidates (latency is measured locally, not a service SLA).
for row in sorted(experiment_rows, key=lambda x: (x["mean_tokens"], -x["strict_recall"])):
    print(f"{row['name']:35s} strict={row['strict_recall']:.3f} tokens={row['mean_tokens']:.1f} latency={row['latency_ms']:.1f}ms")


In [ ]:
def student_release_gate(baseline_summary, challenger_summary, worst_slice,
                         min_strict_gain=0.05, max_token_growth=1.50):
    """Return (passed, reasons) using quality, cost, and worst-slice constraints."""
    reasons = []
    if challenger_summary["strict_recall"] - baseline_summary["strict_recall"] < min_strict_gain:
        reasons.append("strict evidence recall gain is below target")
    if challenger_summary["mean_tokens"] > baseline_summary["mean_tokens"] * max_token_growth:
        reasons.append("mean context tokens exceed growth budget")
    if worst_slice < baseline_summary["strict_recall"] - 0.20:
        reasons.append("a critical slice regressed beyond tolerance")
    return (not reasons), reasons


In [ ]:
# Verification — do not modify.
answerable_types = sorted({r["question_type"] for r in advanced_rows if r["answerable"]})
slice_scores = []
for qtype in answerable_types:
    values = [r["strict_recall"] for r in advanced_rows if r["answerable"] and r["question_type"] == qtype]
    slice_scores.append(statistics.mean(values))
passed, reasons = student_release_gate(baseline_summary, advanced_summary, min(slice_scores))
print("RELEASE GATE:", "PASS" if passed else "FAIL")
for reason in reasons:
    print(" -", reason)
assert isinstance(passed, bool) and isinstance(reasons, list)

# Eval-driven iteration: keep the same retrieval architecture, then tighten candidate and
# context budgets to the Pareto point found above.
OPTIMIZED_ADVANCED = {**ADVANCED, "top_k": 4, "token_budget": 300}
optimized_rows = evaluate_retriever(index, questions, OPTIMIZED_ADVANCED)
optimized_summary = summarize_rows(optimized_rows)
optimized_slices = []
for qtype in answerable_types:
    values = [r["strict_recall"] for r in optimized_rows if r["answerable"] and r["question_type"] == qtype]
    optimized_slices.append(statistics.mean(values))
optimized_passed, optimized_reasons = student_release_gate(
    baseline_summary, optimized_summary, min(optimized_slices)
)
print("\nOPTIMIZED RELEASE GATE:", "PASS" if optimized_passed else "FAIL")
print(" optimized strict recall:", round(optimized_summary["strict_recall"], 3))
print(" optimized mean tokens:", round(optimized_summary["mean_tokens"], 1))
for reason in optimized_reasons:
    print(" -", reason)
assert optimized_passed, optimized_reasons


<h2 id="plan">7.8 — A practical evaluation program</h2>

1. Start with 150–300 expert cases; hold out whole companies/contracts and later amendments.
2. Label minimal evidence spans/pages/cells, atomic required claims, answerability, and for
   numbers the operands/program/unit/period. Include 15–25% unanswerable hard negatives.
3. Establish boundaries: no-context, oracle evidence, full long context, flat RAG, hybrid,
   reranked, structure-aware, and hierarchical/iterative retrieval.
4. Freeze everything except the factor under test. Log parser/chunker/embedder/retriever/
   reranker/router/reader versions plus latency, prompt tokens, cost, and random seed.
5. Validate automated/LLM judges against a double-annotated expert sample and by injected
   failure type. Blind system names and swap pairwise answer order.
6. Ship only behind macro, worst-slice, unsafe-answer, citation, latency, and cost gates.

### Useful public seeds and evaluation research

- [LegalBench-RAG: 6,858 expert-annotated retrieval pairs (2024)](https://arxiv.org/abs/2408.10343)
- [FinanceBench: financial QA with evidence (2023)](https://arxiv.org/abs/2311.11944)
- [TAT-QA: table + text numerical reasoning](https://arxiv.org/abs/2105.07624)
- [FinMRAGBench: multi-page, multi-document multimodal finance (ACL 2026)](https://aclanthology.org/2026.findings-acl.187/)
- [Legal RAG Bench: factorial error decomposition (2026 preprint)](https://arxiv.org/abs/2603.01710)
- [RAGChecker: component-level diagnostics (2024)](https://arxiv.org/abs/2408.08067)
- [ALCE: answer and citation quality (2023)](https://arxiv.org/abs/2305.14627)
- [GaRAGe: grounding, attribution, and deflection (2025)](https://arxiv.org/abs/2506.07671)
- [Lost in the Middle: position sensitivity (TACL 2024)](https://aclanthology.org/2024.tacl-1.9/)

The central rule: evaluate against atomic evidence and provenance, not prose similarity.
That is what makes a miss actionable in a large financial or legal document.
